# Mundial 2026 Predictor — Colab

Este notebook permite ejecutar el predictor desde la web, sin escribir manualmente nombres de países.

Funciones principales:

- Elegir fecha desde un menú.
- Elegir partido desde un menú.
- Generar `match_summary.csv`, `markets_long.csv`, `scorelines_long.csv` y `report.md`.
- Correr una simulación Monte Carlo técnica con checkpoint.

> Proyecto de portafolio de Omar Garcia.


## 1. Clonar repositorio e instalar dependencias

Edita `REPO_URL` si tu repositorio final tiene otro nombre.


In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/omargah/WC26-predictor.git"
REPO_DIR = Path("/content/mundial-2026-predictor")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("Repo:", Path.cwd())


In [ ]:
!pip install -q -r requirements.txt
!pip install -q ipywidgets


## 2. Verificar archivos necesarios

Esta celda revisa si ya existen los archivos de predicción. Si faltan, primero hay que correr el pipeline del repo.


In [ ]:
from pathlib import Path

required = [
    Path("data/predictions/phase03_pending_predictions.csv"),
    Path("models/poisson_dc_base.joblib"),
]

missing = [p for p in required if not p.exists()]

if missing:
    print("Faltan archivos necesarios:")
    for p in missing:
        print(" -", p)
    print("\nEjecuta el pipeline de datos/modelo antes de usar el predictor web.")
else:
    print("OK: archivos principales encontrados.")


## 3. Construir opciones de fechas y partidos

Esto genera listas seguras para dropdowns.


In [ ]:
!python scripts/build_web_reference_options.py


## 4. Predictor por fecha

Selecciona una fecha disponible y ejecuta la predicción.


In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown
from pathlib import Path

dates = pd.read_csv("data/reference/available_dates.csv")

date_dropdown = widgets.Dropdown(
    options=dates["date"].tolist(),
    description="Fecha:",
    layout=widgets.Layout(width="350px"),
)

run_date_button = widgets.Button(
    description="Predecir fecha",
    button_style="success",
)

date_output = widgets.Output()

def run_date_prediction(_):
    with date_output:
        clear_output()
        date = date_dropdown.value
        save_name = f"colab_predictions_by_date_{date}.csv"
        print(f"Calculando partidos de {date}...")
        !python scripts/predict_final_matches.py --date {date} --save-name {save_name}
        folder = Path("data/predictions/formatted") / Path(save_name).stem
        summary = pd.read_csv(folder / "match_summary.csv")
        display(summary)
        display(Markdown((folder / "report.md").read_text(encoding="utf-8")))

run_date_button.on_click(run_date_prediction)

display(date_dropdown, run_date_button, date_output)


## 5. Predictor por partido

Selecciona un partido disponible. No escribas países manualmente.


In [ ]:
matches = pd.read_csv("data/reference/available_matches.csv")

match_dropdown = widgets.Dropdown(
    options=list(zip(matches["dropdown_label"], matches["match"])),
    description="Partido:",
    layout=widgets.Layout(width="650px"),
)

run_match_button = widgets.Button(
    description="Predecir partido",
    button_style="success",
)

match_output = widgets.Output()

def run_match_prediction(_):
    with match_output:
        clear_output()
        match = match_dropdown.value
        safe_name = match.lower().replace(" ", "_").replace("/", "_")
        save_name = f"colab_prediction_{safe_name}.csv"
        print(f"Calculando {match}...")
        !python scripts/predict_final_matches.py --match "{match}" --save-name {save_name}
        folder = Path("data/predictions/formatted") / Path(save_name).stem
        summary = pd.read_csv(folder / "match_summary.csv")
        markets = pd.read_csv(folder / "markets_long.csv")
        scores = pd.read_csv(folder / "scorelines_long.csv")
        display(summary)
        display(markets[markets["rank_within_match"] <= 10])
        display(scores[scores["score_rank_within_match"] <= 10])

run_match_button.on_click(run_match_prediction)

display(match_dropdown, run_match_button, match_output)


## 6. Monte Carlo técnico con checkpoint

Este modo puede tardar. Para Colab se recomienda empezar con pocas simulaciones.


In [ ]:
policy_dropdown = widgets.Dropdown(
    options=["fixed", "resimulate"],
    value="fixed",
    description="Escenario:",
)

n_slider = widgets.IntSlider(
    value=10,
    min=1,
    max=100,
    step=1,
    description="Sims:",
    continuous_update=False,
)

run_mc_button = widgets.Button(
    description="Correr Monte Carlo",
    button_style="warning",
)

mc_output = widgets.Output()

def run_mc(_):
    with mc_output:
        clear_output()
        policy = policy_dropdown.value
        n = n_slider.value
        run_name = f"colab_{policy}_{n}_sims"
        print(f"Corriendo Monte Carlo: {policy}, n={n}")
        !python scripts/simulate_tournament_montecarlo_original_safe_v2.py --played-policy {policy} --n-total {n} --seed 42 --run-name {run_name} --progress-every 1
        folder = Path("data/predictions/mc_original_v2_runs") / run_name
        champs = pd.read_csv(folder / "champion_probabilities.csv")
        rounds = pd.read_csv(folder / "round_probabilities.csv")
        display(champs.head(20))
        display(rounds.head(20))

run_mc_button.on_click(run_mc)

display(policy_dropdown, n_slider, run_mc_button, mc_output)
